In [15]:
import os
import glob
from datasets import load_dataset

# 诊断代码
general_path = "/DATA/disk2/yuhang/.cache/huggingface/hub/datasets--Mxode--Chinese-Instruct/snapshots/1576ffc1e0b21d2980b887de6b5f9abc4a7fb2a8"

print("=== 诊断信息 ===")

# 1. 检查目录是否存在
print(f"目录是否存在: {os.path.exists(general_path)}")

# 2. 列出所有jsonl文件
jsonl_files = glob.glob(f"{general_path}/*/*.jsonl")
print(f"找到的jsonl文件数量: {len(jsonl_files)}")

# 3. 检查前几个文件的状态
for i, file_path in enumerate(jsonl_files[:3]):  # 只检查前3个文件
    print(f"\n文件 {i+1}: {os.path.basename(file_path)}")
    print(f"  - 文件存在: {os.path.exists(file_path)}")
    print(f"  - 文件大小: {os.path.getsize(file_path) if os.path.exists(file_path) else 'N/A'} bytes")
    print(f"  - 是否为软链接: {os.path.islink(file_path)}")
    
    if os.path.islink(file_path):
        link_target = os.readlink(file_path)
        print(f"  - 链接目标: {link_target}")
        
        # 检查绝对路径的链接目标
        if not os.path.isabs(link_target):
            abs_target = os.path.join(os.path.dirname(file_path), link_target)
            print(f"  - 绝对链接目标: {abs_target}")
            print(f"  - 链接目标存在: {os.path.exists(abs_target)}")

# 4. 尝试用pandas直接读取一个文件
try:
    import pandas as pd
    if jsonl_files:
        test_file = jsonl_files[0]
        print(f"\n=== 尝试用pandas读取第一个文件 ===")
        df = pd.read_parquet(test_file)
        print(f"成功读取，数据形状: {df.shape}")
        print(f"列名: {list(df.columns)}")
        if len(df) > 0:
            print(f"第一行数据: {df.iloc[0].to_dict()}")
        else:
            print("文件为空")
except Exception as e:
    print(f"pandas读取失败: {e}")

=== 诊断信息 ===
目录是否存在: True
找到的jsonl文件数量: 15

文件 1: train.jsonl
  - 文件存在: True
  - 文件大小: 151249787 bytes
  - 是否为软链接: True
  - 链接目标: ../../../blobs/f4260e812e2c0910fac6d52f03f7bd30d8640053ce76e90610ea2b1ad8f46f63
  - 绝对链接目标: /DATA/disk2/yuhang/.cache/huggingface/hub/datasets--Mxode--Chinese-Instruct/snapshots/1576ffc1e0b21d2980b887de6b5f9abc4a7fb2a8/disc-law/../../../blobs/f4260e812e2c0910fac6d52f03f7bd30d8640053ce76e90610ea2b1ad8f46f63
  - 链接目标存在: True

文件 2: train.jsonl
  - 文件存在: True
  - 文件大小: 331079592 bytes
  - 是否为软链接: True
  - 链接目标: ../../../blobs/cce11064dadf30140d08e53ec1113b6884543bfc4219efc8a8c3b50fc1105abc
  - 绝对链接目标: /DATA/disk2/yuhang/.cache/huggingface/hub/datasets--Mxode--Chinese-Instruct/snapshots/1576ffc1e0b21d2980b887de6b5f9abc4a7fb2a8/industryinstruction/../../../blobs/cce11064dadf30140d08e53ec1113b6884543bfc4219efc8a8c3b50fc1105abc
  - 链接目标存在: True

文件 3: train.jsonl
  - 文件存在: True
  - 文件大小: 1381853459 bytes
  - 是否为软链接: True
  - 链接目标: ../../../blobs/780018d3dd851417720

In [16]:
import os
from datasets import load_dataset

def resolve_symlinks(file_pattern):
    """解析软链接并返回实际文件路径"""
    import glob
    files = glob.glob(file_pattern)
    resolved_files = []
    
    for file_path in files:
        if os.path.islink(file_path):
            # 解析软链接
            link_target = os.readlink(file_path)
            if not os.path.isabs(link_target):
                # 相对路径，转换为绝对路径
                abs_target = os.path.join(os.path.dirname(file_path), link_target)
            else:
                abs_target = link_target
            
            if os.path.exists(abs_target):
                resolved_files.append(abs_target)
            else:
                print(f"警告: 软链接目标不存在: {abs_target}")
        else:
            resolved_files.append(file_path)
    
    return resolved_files

# 使用解析后的文件路径
general_path = "/DATA/disk2/yuhang/.cache/huggingface/hub/datasets--Mxode--Chinese-Instruct/snapshots/1576ffc1e0b21d2980b887de6b5f9abc4a7fb2a8"
resolved_files = resolve_symlinks(f"{general_path}/*/*.jsonl")

print(f"解析后的文件数量: {len(resolved_files)}")

if resolved_files:
    ds = load_dataset(
        "json",
        data_files=resolved_files,
        split="train"
    )
    print(f"成功加载数据集，大小: {len(ds)}")
else:
    print("没有找到有效的jsonl文件")

解析后的文件数量: 15
成功加载数据集，大小: 4845389


In [18]:
print(ds.column_names)

['id', 'prompt', 'response']


In [12]:
# 检查数据集的列信息
print("数据集列信息:")
print(ds.column_names)
print("\n数据集样本数量:", len(ds))

# 查看几个样本的结构
print("\n前3个样本的结构:")
for i in range(min(3, len(ds))):
    print(f"样本 {i}:")
    for key, value in ds[i].items():
        print(f"  {key}: {str(value)[:100]}...")  # 只显示前100个字符
    print()

# 将数据集转换为LlamaFactory的alpaca格式
def convert_to_alpaca_format(example):
    """
    将Chinese-Instruct数据集转换为alpaca格式
    alpaca格式需要的字段：
    - instruction: 指令/问题
    - input: 输入内容（可选，如果没有可以为空字符串）
    - output: 期望的输出/回答
    """
    return {
        "instruction": example["prompt"],  # 使用prompt作为instruction
        "input": "",  # Chinese-Instruct数据集中prompt已经包含了完整问题，所以input为空
        "output": example["response"]  # 使用response作为output
    }

# 应用转换函数
print("正在转换数据格式...")
alpaca_ds = ds.map(convert_to_alpaca_format,
                   num_proc=30,
                   remove_columns=ds.column_names)

print("转换完成！")
print("转换后的数据集列信息:", alpaca_ds.column_names)

# 查看转换后的样本
print("\n转换后的样本示例:")
for i in range(min(2, len(alpaca_ds))):
    print(f"样本 {i+1}:")
    print(f"  instruction: {alpaca_ds[i]['instruction'][:150]}...")
    print(f"  input: '{alpaca_ds[i]['input']}'")
    print(f"  output: {alpaca_ds[i]['output'][:150]}...")


数据集列信息:
['id', 'prompt', 'response']

数据集样本数量: 4845389

前3个样本的结构:
样本 0:
  id: 1908Czo5lu1R...
  prompt: 请大致描述这段文本的内容
今年4月，西安奔驰女车主薛某引擎盖上维权，一“坐”而红。从“坐出正义”到“反被维权”，这个思路清晰、能笑会哭的女同学，时不时就会上一下热搜，存在感十足。这不，薛同学又有新料曝...
  response: 近日，中国执行信息公开网显示，上海竞集文化发展有限公司因未履行给付义务，公司及实控人徐某因拖欠工资被限制高消费，飞机、动车是坐不了。不仅徐某被“限高”，该公司的法定代表人黄某也被限制高消费。徐某是薛的...

样本 1:
  id: 8Nid6y7NNfoq...
  prompt: 案件：经审理查明:2012年2月24日,原告潘x0与被告黄x2签订《协议书》,双方就江门市新会区五和农场土地(地号分别为:0101030059、0101030060、0101030061、010103...
  response: 双方签订了协议书，具体约定了关于江门市新会区五和农场土地的转让事宜。根据协议的第一条约定，双方约定了转让款项的支付方式及金额。根据第二条约定，乙方应保证合同的合法性并委托资金由第三方代管。根据第四条约...

样本 2:
  id: hj0Il3EvyKpD...
  prompt: 青岛清理拖欠民营企业中小企业账款率已达100%；鼓励“民告官”，行政负责人出庭率去年10月已达100%去年，青岛市加大力度推进清理拖欠民营企业中小企业账款各项工作。去年一年，全市偿还民营中小企业欠款超...
  response: 在去年一年里，青岛市加强工作力度，全力推进清理拖欠民营企业中小企业账款各项工作，完成度已达100%，偿还欠款超过30亿。与此同时，青岛市鼓励支持“民告官”，还要求官员自觉出庭应诉，行政负责人出庭率不断...

正在转换数据格式...


Map (num_proc=30): 100%|██████████| 4845389/4845389 [00:07<00:00, 679324.70 examples/s] 

转换完成！
转换后的数据集列信息: ['instruction', 'input', 'output']

转换后的样本示例:
样本 1:
  instruction: 请大致描述这段文本的内容
今年4月，西安奔驰女车主薛某引擎盖上维权，一“坐”而红。从“坐出正义”到“反被维权”，这个思路清晰、能笑会哭的女同学，时不时就会上一下热搜，存在感十足。这不，薛同学又有新料曝出，这次是“家属”收到了限高令，还是2条！01欠薪1万，“家属”收限高令近日，中国执行信息公开网显示...
  input: ''
  output: 近日，中国执行信息公开网显示，上海竞集文化发展有限公司因未履行给付义务，公司及实控人徐某因拖欠工资被限制高消费，飞机、动车是坐不了。不仅徐某被“限高”，该公司的法定代表人黄某也被限制高消费。徐某是薛的男友。黄某则为薛的母亲。此次被“限高”，金额低到侦探君不敢相信。值得注意的是，曾有人因合同纠纷将上海...
样本 2:
  instruction: 案件：经审理查明:2012年2月24日,原告潘x0与被告黄x2签订《协议书》,双方就江门市新会区五和农场土地(地号分别为:0101030059、0101030060、0101030061、0101030063)转让事宜进行了具体约定其中第一条约定:本协议自签订之日起,甲乙双方应当按照乙方指定账户汇入...
  input: ''
  output: 双方签订了协议书，具体约定了关于江门市新会区五和农场土地的转让事宜。根据协议的第一条约定，双方约定了转让款项的支付方式及金额。根据第二条约定，乙方应保证合同的合法性并委托资金由第三方代管。根据第四条约定，双方约定了不得违约的规定。根据第六条约定，乙方应在规定时间内完成操作并提供土地的公司名。根据第八...


In [19]:
alpaca_ds

Dataset({
    features: ['instruction', 'input', 'output'],
    num_rows: 4845389
})

In [20]:
# 统计instruction字段的长度并计算中位数
import numpy as np

print("正在统计instruction字段的长度...")

# 计算每个样本的instruction长度
instruction_lengths = []
for example in alpaca_ds:
    instruction_lengths.append(len(example['instruction']))

# 转换为numpy数组以便计算统计信息
instruction_lengths = np.array(instruction_lengths)

# 计算统计信息
mean_length = np.mean(instruction_lengths)
median_length = np.median(instruction_lengths)
min_length = np.min(instruction_lengths)
max_length = np.max(instruction_lengths)
std_length = np.std(instruction_lengths)

print(f"\ninstruction字段长度统计:")
print(f"  总样本数: {len(instruction_lengths):,}")
print(f"  平均长度: {mean_length:.2f} 字符")
print(f"  中位数长度: {median_length:.2f} 字符")
print(f"  最短长度: {min_length} 字符")
print(f"  最长长度: {max_length} 字符")
print(f"  标准差: {std_length:.2f}")

# 计算不同长度区间的分布
print(f"\n长度分布:")
ranges = [(0, 50), (50, 100), (100, 200), (200, 500), (500, 1000), (1000, float('inf'))]
for start, end in ranges:
    if end == float('inf'):
        count = np.sum(instruction_lengths >= start)
        print(f"  {start}+ 字符: {count:,} 样本 ({count/len(instruction_lengths)*100:.2f}%)")
    else:
        count = np.sum((instruction_lengths >= start) & (instruction_lengths < end))
        print(f"  {start}-{end-1} 字符: {count:,} 样本 ({count/len(instruction_lengths)*100:.2f}%)")


正在统计instruction字段的长度...

instruction字段长度统计:
  总样本数: 4,845,389
  平均长度: 81.06 字符
  中位数长度: 47.00 字符
  最短长度: 0 字符
  最长长度: 21341 字符
  标准差: 150.46

长度分布:
  0-49 字符: 2,493,138 样本 (51.45%)
  50-99 字符: 1,265,887 样本 (26.13%)
  100-199 字符: 758,018 样本 (15.64%)
  200-499 字符: 275,739 样本 (5.69%)
  500-999 字符: 36,308 样本 (0.75%)
  1000+ 字符: 16,299 样本 (0.34%)


In [21]:
# 筛选出instruction字段长度大于等于500的数据
print("正在筛选instruction字段长度大于等于500的数据...")

# 创建一个函数来判断instruction长度是否大于等于500
def is_long_instruction(example):
    return len(example['instruction']) >= 500

# 使用filter方法筛选出长instruction的数据
alpaca_ds_long = alpaca_ds.filter(is_long_instruction, num_proc=30)

print(f"筛选完成！")
print(f"原始数据集大小: {len(alpaca_ds):,} 条样本")
print(f"长instruction数据集大小: {len(alpaca_ds_long):,} 条样本")
print(f"长instruction数据占比: {len(alpaca_ds_long)/len(alpaca_ds)*100:.2f}%")

# 同时更新原始数据集，移除长instruction的数据
print("\n正在从原始数据集中移除长instruction数据...")

# 创建一个函数来判断instruction长度是否小于500
def is_short_instruction(example):
    return len(example['instruction']) < 500

# 更新原始数据集，只保留instruction长度小于500的数据
alpaca_ds = alpaca_ds.filter(is_short_instruction, num_proc=30)

print(f"更新后的原始数据集大小: {len(alpaca_ds):,} 条样本")
print(f"移除的长instruction数据: {len(alpaca_ds_long):,} 条样本")

# 验证数据完整性
print(f"\n数据完整性验证:")
print(f"原始数据集(短instruction) + 长instruction数据集 = {len(alpaca_ds) + len(alpaca_ds_long):,}")
print(f"这应该等于原始总数据量")


正在筛选instruction字段长度大于等于500的数据...


Filter (num_proc=30): 100%|██████████| 4845389/4845389 [00:01<00:00, 3734037.76 examples/s]

筛选完成！
原始数据集大小: 4,845,389 条样本
长instruction数据集大小: 52,607 条样本
长instruction数据占比: 1.09%

正在从原始数据集中移除长instruction数据...



Filter (num_proc=30): 100%|██████████| 4845389/4845389 [00:01<00:00, 3797769.53 examples/s]

更新后的原始数据集大小: 4,792,782 条样本
移除的长instruction数据: 52,607 条样本

数据完整性验证:
原始数据集(短instruction) + 长instruction数据集 = 4,845,389
这应该等于原始总数据量


In [30]:
alpaca_ds[1000]

{'instruction': "2015年9月份，被告人韩四杰及牛某、陆某（均已判决）等人，伪造订货单，虚构从某外贸公司接到一批订单的事实，以黄某（另案处理）的名义陆续向被害人张某1采购价值人民币128160元的大鸭夹、爪夹、发簪等头饰。被害人张某1交付上述货物后，被告人韩四杰及牛某、陆某等人随即将该批货物分多次低价出售给五爱库存商，所得货款除人民币8800元支付给被害人张某1外，其余均用于个人挥霍。现被告人韩四杰已退赔被害人张某1损失人民币30000元并取得谅解。另查明同案犯牛某、陆某、黄某均已分别退赔被害人张某1人民币30000元。\n以上是一起案件，请识别出所有的事件触发词和事件类型\n例如，事件触发词'撞击'对应事件类型'交通事故'。请用JSON数组输出结果，数组中的每个元素也是一个数组，形如：[事件触发词, 事件类型]。",
 'input': '',
 'output': '[\n    ["伪造", "伪造"],\n    ["虚构", "伪造"],\n    ["采购", "买入"],\n    ["交付", "支付/给付"],\n    ["出售", "卖出"],\n    ["支付", "支付/给付"],\n    ["挥霍", "销赃"],\n    ["退", "退赃"],\n    ["赔", "赔偿"],\n    ["损失", "受损"],\n    ["谅解", "谅解"]\n]'}

In [32]:
# 对数据集进行shuffle操作
print("正在对数据集进行shuffle操作...")

# 使用shuffle方法打乱数据集顺序
# seed参数确保结果可重现
alpaca_ds = alpaca_ds.shuffle(seed=42)

print("数据集shuffle完成！")
print(f"shuffle后的数据集大小: {len(alpaca_ds):,} 条样本")

# 查看shuffle后的第一个样本
print("\nshuffle后的第一个样本:")
print(f"instruction: {alpaca_ds[0]['instruction'][:100]}...")
print(f"input: '{alpaca_ds[0]['input']}'")
print(f"output: {alpaca_ds[0]['output'][:100]}...")


正在对数据集进行shuffle操作...
数据集shuffle完成！
shuffle后的数据集大小: 4,792,782 条样本

shuffle后的第一个样本:
instruction: 可以缓解高血压引起的眼底出血吗，我妈在前两年就患高血压疾病，自从我妈换上高血压疾病之后，不仅经常出现头晕，并且最近这段时间还再次出现了许多眼部症状，这两天在医院实施仔细检查，发觉我妈再次出现了眼底出血...
input: ''
output: 您母亲的情况确实需要综合管理，以控制血压并尽可能减少并发症的风险。高血压引起的眼底出血是一种比较严重的并发症，需要及时治疗。以下是一些基本的建议：

1. **药物治疗**：首先，需要通过药物控制血压...


In [38]:
# 统计alpaca_ds中instruction为空的样本
print("正在统计instruction为空的样本...")

# 创建检查函数，判断instruction是否为空
def is_empty_instruction(example):
    """检查instruction是否为空"""
    instruction = example['instruction']
    # 检查是否为空字符串、None或只包含空白字符
    return not instruction or instruction.strip() == ""

# 过滤出instruction为空的样本
empty_instruction_samples = alpaca_ds.filter(is_empty_instruction, num_proc=30)

print(f"\n统计结果:")
print(f"数据集总样本数: {len(alpaca_ds):,} 条")
print(f"instruction为空的样本数: {len(empty_instruction_samples):,} 条")
print(f"空instruction样本占比: {len(empty_instruction_samples)/len(alpaca_ds)*100:.2f}%")
print(f"有效instruction样本数: {len(alpaca_ds) - len(empty_instruction_samples):,} 条")

# 如果有空instruction样本，展示几个例子
if len(empty_instruction_samples) > 0:
    print(f"\n展示前3个空instruction样本:")
    for i in range(min(3, len(empty_instruction_samples))):
        sample = empty_instruction_samples[i]
        print(f"样本 {i+1}:")
        print(f"  instruction: '{sample['instruction']}'")
        print(f"  input: '{sample['input']}'")
        print(f"  output: {sample['output'][:100]}...")
        print()
else:
    print("\n✅ 没有发现instruction为空的样本！")

print("instruction为空样本统计完成！")



正在统计instruction为空的样本...


Filter (num_proc=30): 100%|██████████| 4792782/4792782 [00:07<00:00, 675608.01 examples/s]


统计结果:
数据集总样本数: 4,792,782 条
instruction为空的样本数: 7 条
空instruction样本占比: 0.00%
有效instruction样本数: 4,792,775 条

展示前3个空instruction样本:
样本 1:
  instruction: ''
  input: ''
  output: 1. {"content":"电气设备的标准安装高度是多少？", "summary":"根据《通用安装工程工程量计算规范》第4.2.7条，电气设备的标准安装高度为5米。"}
2. {"content"...

样本 2:
  instruction: ''
  input: ''
  output: **Conversation 1: Aranha's Encounter at a Social Gathering**

**Aranha:** *(Gliding through the crow...

样本 3:
  instruction: ''
  input: ''
  output: Please note that the provided queries are based on an assumed database schema. You may need to adjus...

instruction为空样本统计完成！


#### 保存Instruction字段长度小于500的样本

In [33]:
# 保存转换后的数据集到本地磁盘
import os

# 定义保存路径
save_path = "/DATA/disk2/yuhang/.cache/bit_brain_data/sft_llamafactory/chinese-instruct"

# 创建目录（如果不存在）
os.makedirs(save_path, exist_ok=True)

print(f"正在保存数据集到: {save_path}")

# 保存为jsonl格式，这是LlamaFactory常用的格式
alpaca_ds.to_json(os.path.join(save_path, "chinese_instruct.jsonl"), 
                  orient="records", 
                  lines=True, 
                  force_ascii=False)

print("数据集保存完成！")
print(f"保存位置: {save_path}/chinese_instruct.jsonl")
print(f"数据集大小: {len(alpaca_ds)} 条样本")

# 验证保存的文件
import json
print("\n验证保存的文件...")
with open(os.path.join(save_path, "chinese_instruct.jsonl"), 'r', encoding='utf-8') as f:
    # 读取前几行验证格式
    for i, line in enumerate(f):
        if i >= 2:  # 只验证前2行
            break
        sample = json.loads(line)
        print(f"验证样本 {i+1}:")
        print(f"  instruction: {sample['instruction'][:100]}...")
        print(f"  input: '{sample['input']}'")
        print(f"  output: {sample['output'][:100]}...")
        print()

print("文件验证完成，格式正确！")


正在保存数据集到: /DATA/disk2/yuhang/.cache/bit_brain_data/sft_llamafactory/chinese-instruct


Creating json from Arrow format: 100%|██████████| 4793/4793 [02:01<00:00, 39.53ba/s]

数据集保存完成！
保存位置: /DATA/disk2/yuhang/.cache/bit_brain_data/sft_llamafactory/chinese-instruct/chinese_instruct.jsonl
数据集大小: 4792782 条样本

验证保存的文件...
验证样本 1:
  instruction: 可以缓解高血压引起的眼底出血吗，我妈在前两年就患高血压疾病，自从我妈换上高血压疾病之后，不仅经常出现头晕，并且最近这段时间还再次出现了许多眼部症状，这两天在医院实施仔细检查，发觉我妈再次出现了眼底出血...
  input: ''
  output: 您母亲的情况确实需要综合管理，以控制血压并尽可能减少并发症的风险。高血压引起的眼底出血是一种比较严重的并发症，需要及时治疗。以下是一些基本的建议：

1. **药物治疗**：首先，需要通过药物控制血压...

验证样本 2:
  instruction: 对话一位新手妈妈，给出新生儿护理的建议。...
  input: ''
  output: 小明：你好，我听说你刚刚成为了一位新手妈妈，恭喜恭喜！
新手妈妈：谢谢，我确实是个新手妈妈。我现在有些担心怎么照顾我的新生儿。
小明：没关系，我可以给你提供一些建议。首先要保证新生儿的饮食和睡眠。新生...

文件验证完成，格式正确！


#### 保存"instruction"中长度大于500的长输入样本

In [35]:
len(alpaca_ds_long[1]['instruction'])

950

In [37]:
# 保存转换后的数据集到本地磁盘
import os

# 定义保存路径
save_path = "/DATA/disk2/yuhang/.cache/bit_brain_data/sft_llamafactory/chinese-instruct"

# 创建目录（如果不存在）
os.makedirs(save_path, exist_ok=True)

print(f"正在保存数据集到: {save_path}")

# 保存为jsonl格式，这是LlamaFactory常用的格式
alpaca_ds_long.to_json(os.path.join(save_path, "chinese_instruct_long.jsonl"), 
                  orient="records", 
                  lines=True, 
                  force_ascii=False)

print("数据集保存完成！")
print(f"保存位置: {save_path}/chinese_instruct_long.jsonl")
print(f"数据集大小: {len(alpaca_ds)} 条样本")

# 验证保存的文件
import json
print("\n验证保存的文件...")
with open(os.path.join(save_path, "chinese_instruct_long.jsonl"), 'r', encoding='utf-8') as f:
    # 读取前几行验证格式
    for i, line in enumerate(f):
        if i >= 2:  # 只验证前2行
            break
        sample = json.loads(line)
        print(f"验证样本 {i+1}:")
        print(f"  instruction: {sample['instruction'][:100]}...")
        print(f"  input: '{sample['input']}'")
        print(f"  output: {sample['output'][:100]}...")
        print()

print("文件验证完成，格式正确！")


正在保存数据集到: /DATA/disk2/yuhang/.cache/bit_brain_data/sft_llamafactory/chinese-instruct


Creating json from Arrow format: 100%|██████████| 53/53 [00:02<00:00, 20.24ba/s]

数据集保存完成！
保存位置: /DATA/disk2/yuhang/.cache/bit_brain_data/sft_llamafactory/chinese-instruct/chinese_instruct_long.jsonl
数据集大小: 4792782 条样本

验证保存的文件...
验证样本 1:
  instruction: 请大致描述这段文本的内容
今年4月，西安奔驰女车主薛某引擎盖上维权，一“坐”而红。从“坐出正义”到“反被维权”，这个思路清晰、能笑会哭的女同学，时不时就会上一下热搜，存在感十足。这不，薛同学又有新料曝...
  input: ''
  output: 近日，中国执行信息公开网显示，上海竞集文化发展有限公司因未履行给付义务，公司及实控人徐某因拖欠工资被限制高消费，飞机、动车是坐不了。不仅徐某被“限高”，该公司的法定代表人黄某也被限制高消费。徐某是薛的...

验证样本 2:
  instruction: 案件：经审理查明:2012年2月24日,原告潘x0与被告黄x2签订《协议书》,双方就江门市新会区五和农场土地(地号分别为:0101030059、0101030060、0101030061、010103...
  input: ''
  output: 双方签订了协议书，具体约定了关于江门市新会区五和农场土地的转让事宜。根据协议的第一条约定，双方约定了转让款项的支付方式及金额。根据第二条约定，乙方应保证合同的合法性并委托资金由第三方代管。根据第四条约...

文件验证完成，格式正确！
